In [9]:
import os
import scipy.io.wavfile
import matplotlib.pyplot as plt
import numpy as np
from scipy.fftpack import fft
import copy

CONCERT_PITCH = 440
ALL_NOTES = ["A", "A#", "B", "C", "C#", "D", "D#", "E", "F", "F#", "G", "G#"]
NUM_HPS = 4  # Số lượng hài âm để tính HPS
ACCURACY_THRESHOLD = 5  # Ngưỡng chính xác (đơn vị: cents)

def find_closest_note(pitch):
    i = int(np.round(np.log2(pitch / CONCERT_PITCH) * 12))
    closest_note = ALL_NOTES[i % 12] + str(4 + (i + 9) // 12)
    closest_pitch = CONCERT_PITCH * 2 ** (i / 12)
    cents = 1200 * np.log2(pitch / closest_pitch)
    return closest_note, closest_pitch, cents

def extract_cents_from_filename(filename):
    """Lấy số cents và hướng lệch từ tên file"""
    if "_high_" in filename:
        direction = 1  # Cao
        parts = filename.split("_high_")
    elif "_low_" in filename:
        direction = -1  # Thấp
        parts = filename.split("_low_")
    else:
        raise ValueError(f"Không tìm thấy thông tin lệch cents trong tên file: {filename}")
    cents = int(parts[1].replace("cents.wav", ""))
    return direction * cents

def process_folder(folder_path):
    """Kiểm tra các file trong folder"""
    results = []
    total_difference = 0  # Tổng chênh lệch
    total_files = 0  # Tổng số file
    accurate_files = 0  # Số file chính xác

    for filename in os.listdir(folder_path):
        if filename.endswith(".wav"):
            file_path = os.path.join(folder_path, filename)
            # Lấy giá trị cents từ tên file
            try:
                expected_cents = extract_cents_from_filename(filename)
            except ValueError as e:
                print(e)
                continue

            # Đọc file và tính giá trị lệch cents thực tế
            sampleFreq, myRecording = scipy.io.wavfile.read(file_path)
            if len(myRecording.shape) > 1:  # Nếu là stereo
                myRecording = myRecording.mean(axis=1)
            myRecording = myRecording.astype(float)
            myRecording = myRecording / np.max(np.abs(myRecording))

            # FFT và tính tần số cơ bản
            window = np.hanning(len(myRecording))
            windowed_signal = myRecording * window
            absFreqSpectrum = abs(fft(windowed_signal))
            freqSpectrum = absFreqSpectrum[: len(myRecording) // 2]
            mag_spec_ipol = np.interp(
                np.arange(0, len(freqSpectrum), 1 / NUM_HPS),
                np.arange(0, len(freqSpectrum)),
                freqSpectrum,
            )
            mag_spec_ipol = mag_spec_ipol / np.linalg.norm(mag_spec_ipol, ord=2)
            hps_spec = copy.deepcopy(mag_spec_ipol)
            for i in range(NUM_HPS):
                tmp_hps_spec = np.multiply(
                    hps_spec[: int(np.ceil(len(mag_spec_ipol) / (i + 1)))],
                    mag_spec_ipol[:: (i + 1)],
                )
                if not any(tmp_hps_spec):
                    break
                hps_spec = tmp_hps_spec
            max_ind = np.argmax(hps_spec)
            max_freq = max_ind * (sampleFreq / len(myRecording)) / NUM_HPS

            # Tìm nốt và độ lệch cents
            _, _, detected_cents = find_closest_note(max_freq)

            # So sánh với giá trị trong tên file
            difference = abs(expected_cents - detected_cents)
            is_accurate = difference <= ACCURACY_THRESHOLD
            if is_accurate:
                accurate_files += 1

            total_difference += difference
            total_files += 1

            results.append({
                "file": filename,
                "expected_cents": expected_cents,
                "detected_cents": detected_cents,
                "difference": difference,
                "is_accurate": is_accurate,
            })

    # Tính trung bình và độ chính xác tổng thể
    average_difference = total_difference / total_files if total_files > 0 else 0
    accuracy_percentage = (accurate_files / total_files * 100) if total_files > 0 else 0

    return results, average_difference, accuracy_percentage

# Gọi hàm và in kết quả
FOLDER_PATH = r"D:\\MSE23\\XLTHS\\Musical Instrument Tuning\\archive\\Guitar Dataset\\E2_Test_SF"
results, average_difference, accuracy_percentage = process_folder(FOLDER_PATH)

for result in results:
    print(f"File: {result['file']}")
    print(f"Dự đoán: {result['expected_cents']} cents")
    print(f"Phát hiện: {result['detected_cents']:.2f} cents")
    print(f"Chênh lệch: {result['difference']:.2f} cents")
    print(f"Chính xác: {'✅' if result['is_accurate'] else '❌'}\n")

print(f"Chênh lệch trung bình: {average_difference:.2f} cents")
print(f"Độ chính xác trung bình: {accuracy_percentage:.2f}%")

File: E2-1-spn_high_25cents.wav
Dự đoán: 25 cents
Phát hiện: 30.36 cents
Chênh lệch: 5.36 cents
Chính xác: ❌

File: E2-1-spn_low_11cents.wav
Dự đoán: -11 cents
Phát hiện: -5.64 cents
Chênh lệch: 5.36 cents
Chính xác: ❌

File: E2-10-sfn_high_18cents.wav
Dự đoán: 18 cents
Phát hiện: 18.08 cents
Chênh lệch: 0.08 cents
Chính xác: ✅

File: E2-10-sfn_low_11cents.wav
Dự đoán: -11 cents
Phát hiện: -10.92 cents
Chênh lệch: 0.08 cents
Chính xác: ✅

File: E2-11-sfm_high_20cents.wav
Dự đoán: 20 cents
Phát hiện: 22.71 cents
Chênh lệch: 2.71 cents
Chính xác: ✅

File: E2-11-sfm_low_10cents.wav
Dự đoán: -10 cents
Phát hiện: -7.29 cents
Chênh lệch: 2.71 cents
Chính xác: ✅

File: E2-12-snn_high_21cents.wav
Dự đoán: 21 cents
Phát hiện: 18.46 cents
Chênh lệch: 2.54 cents
Chính xác: ✅

File: E2-12-snn_low_16cents.wav
Dự đoán: -16 cents
Phát hiện: -18.55 cents
Chênh lệch: 2.55 cents
Chính xác: ✅

File: E2-13-spn_high_25cents.wav
Dự đoán: 25 cents
Phát hiện: 35.60 cents
Chênh lệch: 10.60 cents
Chính xác: ❌

